<a href="https://colab.research.google.com/github/abrandonwang/text-to-sound/blob/main/fsd50k_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%cd /content/fsd
!rm -f FSD50K.dev_audio.*          # deletes all partial / renamed parts (keeps ground_truth)
!ls                                # should show only FSD50K.ground_truth.zip and urls.txt
!aria2c -x 8 -s 8 -j 6 --summary-interval=30 --auto-file-renaming=false --allow-overwrite=true -i urls.txt

/content/fsd
FSD50K.ground_truth.zip  urls.txt

11/08 03:28:01 [NOTICE] Downloading 6 item(s)
 *** Download Progress Summary as of Sun Nov  8 03:28:31 2026 *** 
=
[#6a0d8b 176MiB/2.1GiB(8%) CN:8 DL:8.8MiB ETA:3m47s]
FILE: /content/fsd/FSD50K.dev_audio.zip
-
[#262269 323MiB/3.0GiB(10%) CN:8 DL:16MiB ETA:2m51s]
FILE: /content/fsd/FSD50K.dev_audio.z01
-
[#19b7fb 318MiB/3.0GiB(10%) CN:8 DL:15MiB ETA:2m56s]
FILE: /content/fsd/FSD50K.dev_audio.z02
-
[#7aa7d1 317MiB/3.0GiB(10%) CN:8 DL:19MiB ETA:2m22s]
FILE: /content/fsd/FSD50K.dev_audio.z03
-
[#fa19cc 191MiB/3.0GiB(6%) CN:8 DL:10MiB ETA:4m25s]
FILE: /content/fsd/FSD50K.dev_audio.z04
-
[#db486c 423MiB/3.0GiB(13%) CN:8 DL:16MiB ETA:2m42s]
FILE: /content/fsd/FSD50K.dev_audio.z05
-

 *** Download Progress Summary as of Sun Nov  8 03:29:03 2026 *** 
=
[#6a0d8b 692MiB/2.1GiB(31%) CN:8 DL:15MiB ETA:1m36s]
FILE: /content/fsd/FSD50K.dev_audio.zip
-
[#262269 0.9GiB/3.0GiB(30%) CN:8 DL:17MiB ETA:2m]
FILE: /content/fsd/FSD50K.dev_audio.z01
-
[#19b7fb 76

In [3]:
%cd /content/fsd
!zip -q -s 0 FSD50K.dev_audio.zip --out dev_full.zip
!rm FSD50K.dev_audio.z0* FSD50K.dev_audio.zip
!unzip -q dev_full.zip && rm dev_full.zip
!unzip -q FSD50K.ground_truth.zip
!ls FSD50K.dev_audio | wc -l
!head -3 FSD50K.ground_truth/dev.csv
!df -h /content

/content/fsd
40966
fname,labels,mids,split
64760,"Electric_guitar,Guitar,Plucked_string_instrument,Musical_instrument,Music","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train
16399,"Electric_guitar,Guitar,Plucked_string_instrument,Musical_instrument,Music","/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf",train
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   25G   89G  22% /


In [4]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/text-to-sound/"
CKPT = DRIVE + "checkpoints/"
OUT = DRIVE + "data/fsd/"
import os
os.makedirs(OUT, exist_ok = True)

import numpy as np, pandas as pd, torch, librosa
import torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import Audio, display
!test -d BigVGAN || git clone -q https://github.com/NVIDIA/BigVGAN
!sed -i 's/proxies: Optional\[Dict\],/proxies: Optional[Dict] = None,/; s/resume_download: bool,/resume_download: bool = False,/' BigVGAN/bigvgan.py
import sys; sys.path.insert(0, "BigVGAN")
import bigvgan
from meldataset import get_mel_spectrogram
from transformers import ClapModel, ClapProcessor
%matplotlib inline

device = "cuda"
vocoder = bigvgan.BigVGAN.from_pretrained("nvidia/bigvgan_v2_22khz_80band_256x", use_cuda_kernel = False)
vocoder.remove_weight_norm()
vocoder = vocoder.eval().to(device)
h = vocoder.h
MIN, MAX = [float(v) for v in np.load(DRIVE + "data/mel_stats.npy")]
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").eval().to(device)
proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
print(MIN, MAX)

Mounted at /content/drive


config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights from nvidia/bigvgan_v2_22khz_80band_256x


bigvgan_generator.pt: reconstructing file:   0%|          |  0.00B /  449MB            

bigvgan_generator.pt: downloading bytes:           |  0.00B            

... (10 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

-11.512925148010254 2.4254610538482666


In [5]:
class Autoencoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = nn.Sequential(
      nn.Conv2d(1, 64, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(64, 64, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(64, 128, 3, 2, 1),
      nn.SiLU(),
      nn.Conv2d(128, 128, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(128, 256, 3, 2, 1),
      nn.SiLU(),
      nn.Conv2d(256, 256, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(256, 8, 3, 1, 1),
    )
    self.decoder = nn.Sequential(
        nn.Conv2d(4, 256, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(256, 256, 3, 1, 1),
        nn.SiLU(),
        nn.Upsample(scale_factor = 2),
        nn.Conv2d(256, 128, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(128, 128, 3, 1, 1),
        nn.SiLU(),
        nn.Upsample(scale_factor = 2),
        nn.Conv2d(128, 64, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(64, 64, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(64, 1, 3, 1, 1),
    )
  def forward(self, x):
    mu, logvar = self.encoder(x).chunk(2, dim = 1)
    z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
    return self.decoder(z), mu, logvar

vae = Autoencoder().to(device)
vae.load_state_dict(torch.load(CKPT + "vae_gan2.pt", map_location = device))
vae.eval()

Autoencoder(
  (encoder): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): SiLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): SiLU()
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): SiLU()
    (6): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): SiLU()
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (9): SiLU()
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): SiLU()
    (12): Conv2d(256, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (decoder): Sequential(
    (0): Conv2d(4, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): SiLU()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): SiLU()
    (4): Upsample(scale_factor=2.0, mode='nearest')
    (5): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), paddin

In [ ]:
df = pd.read_csv("/content/fsd/FSD50K.ground_truth/dev.csv")
AUDIO = "/content/fsd/FSD50K.dev_audio/"
CROP, MAX_CROPS, SILENCE, SHARD = 65536, 3, 0.01, 2000

def captions_for(labels):
    names = [l.replace("_", " ").lower() for l in labels.split(",")]
    a = names[0]
    b = names[1] if len(names) > 1 else names[0]
    return [", ".join(names), a, f"the sound of {a}", f"a recording of {a} and {b}", f"{a} sound"]

text_cache = {}
def text_embs(labels):
    if labels not in text_cache:
        with torch.inference_mode():
            t_in = proc(text = captions_for(labels), return_tensors = "pt", padding = True).to(device)
            text_cache[labels] = clap.get_text_features(**t_in).pooler_output.cpu()
    return text_cache[labels]                                   # [5, 512]

def crops_of(path):
    wav, _ = librosa.load(path, sr = 22050, mono = True)
    wav = np.clip(wav, -1, 1)
    if len(wav) < CROP:                                         # short clip: pad with silence
        if np.sqrt(np.mean(wav ** 2)) < SILENCE:
            return []
        return [np.pad(wav, (0, CROP - len(wav)))]
    out = []
    for s in range(0, len(wav) - CROP + 1, CROP):
        c = wav[s:s + CROP]
        if np.sqrt(np.mean(c ** 2)) >= SILENCE:
            out.append(c)
        if len(out) == MAX_CROPS:
            break
    return out

@torch.no_grad()
def encode(mels):
    x = torch.stack(mels)                                       # [n, 1, 80, 256]
    x = ((x - MIN) / (MAX - MIN) * 2 - 1).clamp(-1, 1)          # Clotho's scaling
    lats = []
    for i in range(0, len(x), 64):
        mu, _ = vae.encoder(x[i:i + 64].to(device)).chunk(2, dim = 1)
        lats.append(mu.cpu().half())
    return torch.cat(lats)

END = 2000                                                      # first run: only shard 0, as a test
for k in range(0, min(END, len(df)), SHARD):
    path_out = OUT + f"shard_{k // SHARD:03d}.npz"
    if os.path.exists(path_out):
        continue
    mels, audio_embs, texts, names = [], [], [], []
    rows = df.iloc[k:k + SHARD]
    for r in tqdm(rows.itertuples(), total = len(rows), desc = f"shard {k // SHARD}"):
        try:
            crops = crops_of(AUDIO + f"{r.fname}.wav")
        except Exception as e:
            print("skip", r.fname, e)
            continue
        if not crops:
            continue
        for c in crops:
            mels.append(get_mel_spectrogram(torch.tensor(c).unsqueeze(0), h))
        c48 = [librosa.resample(c, orig_sr = 22050, target_sr = 48000) for c in crops]
        with torch.inference_mode():
            a_in = proc(audio = c48, return_tensors = "pt", sampling_rate = 48000).to(device)
            audio_embs.append(clap.get_audio_features(**a_in).pooler_output.cpu())
        texts += [text_embs(r.labels)] * len(crops)
        names += [r.fname] * len(crops)
    np.savez(path_out,
             latents = encode(mels).numpy(),
             clap_audio = torch.cat(audio_embs).half().numpy(),
             clap_text = torch.stack(texts).half().numpy(),
             fname = np.array(names))
    print("saved", path_out, len(names), "crops")

shard 0:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
d = np.load(OUT + "shard_000.npz")
print({k: d[k].shape for k in d})
with torch.no_grad():
    z = torch.tensor(d["latents"][:4].astype("float32")).to(device)
    mel = vae.decoder(z).clamp(-1, 1)
    wav = vocoder((((mel + 1) / 2) * (MAX - MIN) + MIN).squeeze(1))
for i in range(4):
    print(df[df.fname == int(d["fname"][i])].labels.values[0])
    display(Audio(wav[i].squeeze().cpu().numpy(), rate = 22050))